# GPT2

1. https://benhay.es/posts/building-gpt2/
2. https://github.com/tinygrad/tinygrad/blob/master/examples/gpt2.py
3. https://github.com/karpathy/minGPT


In [1]:
import math
import torch
from torch import nn

In [13]:
class MHA(nn.Module):
    def __init__(self, dim:int, n_heads:int):
        super().__init__()
        self.n_heads = n_heads

        self.attn_q = torch.nn.Linear(dim, dim)
        self.attn_k = torch.nn.Linear(dim, dim)
        self.attn_v = torch.nn.Linear(dim, dim)
        self.attn_o = torch.nn.Linear(dim, dim)

    def forward(self, x:torch.Tensor, causal:bool=True) -> torch.Tensor:
        # attention projections
        q = self.attn_q(x) # (B,T,D) -> (B,T,D)
        k = self.attn_k(x) # (B,T,D) -> (B,T,D)
        v = self.attn_v(x) # (B,T,D) -> (B,T,D)

        # reshape
        B, T, D = x.shape
        q = q.reshape(B, T, self.n_heads, -1).transpose(1, 2) # (B,T,D) -> (B,T,H,Dh) -> (B,H,T,Dh)
        k = k.reshape(B, T, self.n_heads, -1).transpose(1, 2) # (B,T,D) -> (B,T,H,Dh) -> (B,H,T,Dh)
        v = v.reshape(B, T, self.n_heads, -1).transpose(1, 2) # (B,T,D) -> (B,T,H,Dh) -> (B,H,T,Dh)

        # rope

        # kv cache

        # mask

        # attention score
        qk = q @ k.transpose(-1, -2) / math.sqrt(D) # (B,H,T,Dh) (B,H,Dh,T) -> (B,H,T,T)
        sim = qk.softmax(-1) # (B,H,T,T) -> (B,H,T,T)
        score = qk @ v # (B,H,T,T) (B,H,T,Dh) -> (B,H,T,Dh)
        score = score.transpose(1,2).reshape(B,T,-1) # (B,H,T,Dh) -> (B,T,H,Dh) -> (B,T,H*Dh) = (B,T,D)

        # out
        out = self.attn_o(score) # (B,T,D) (D,D) -> (B,T,D)
        return out


In [14]:
class FFN(nn.Module):
    def __init__(self, dim1:int, dim2:int):
        super().__init__()
        self.l1 = torch.nn.Linear(dim1, dim2)
        self.l2 = torch.nn.Linear(dim2, dim1)
        self.act = torch.nn.ReLU()

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        out = self.l1(x)        # (B,T,D1) -> (B,T,D2)
        out = self.act(out)    # (B,T,D2) -> (B,T,D2)
        out = self.l2(out)      # (B,T,D2) -> (B,T,D1)
        return out

In [15]:
class TransformerBlock(nn.Module):
    def __init__(self, dim:int, n_heads:int):
        super().__init__()
        self.attn = MHA(dim, n_heads)
        self.ffn = FFN(dim, dim*4)
        self.attn_norm = nn.LayerNorm(dim)
        self.ffn_norm = nn.LayerNorm(dim)

    def forward(self, x:torch.Tensor) -> torch.Tensor:
        x = self.attn(self.attn_norm(x)) + x
        x = self.ffn(self.ffn_norm(x)) + x
        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, dim:int, n_heads:int, n_layers:int, vocab_size:int):
        super().__init__()
        self.layers = [TransformerBlock(dim, n_heads) for _ in range(n_layers)]
        self.vocab_embd = nn.Embedding(vocab_size, dim)
        self.out_norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size)

    def forward(self, x):
        x = self.vocab_embd(x) # (B,T) -> (B,T,D)
        for layer in self.layers:
            x = layer(x) # (B,T,D) -> (B,T,D)
        x = self.out_norm(x) # (B,T,D) -> (B,T,D)
        x = self.lm_head(x) # (B,T,D) -> (B,T,V)
        return x

    @staticmethod
    def generate(text, max_new_tokens:int):
        token_ids = torch.arange(10).reshape(2, 5) # (B,T)

        for _ in range(max_new_tokens):
            logits = self(token_ids) # (B,T) -> (B,T,V)
            last_logits = logits[:, -1, :].squeeze() # (B,T,V) -> (B,V)
            probs = last_logits.softmax(-1)
            next_token_id = torch.multinomial(probs) # (B,V) -> (B,1)
            token_ids = torch.cat((token_ids, next_token_id), dim=1) # (B,T) (B,1) -> (B,T+1)
        return token_ids # (B,T+max_new_tokens)

In [17]:
B,T,D,H,L,V = 2, 5, 16, 4, 3, 256

model = Transformer(D,H,L,V)
x = torch.arange(B*T).reshape(B,T)

In [19]:
out = model(x)